

```
vcbs_report_ingestion.ipynb          miniRAG_1.ipynb
─────────────────────────            ───────────────
Fetch API                           Load vcbs_reports.json
Filter + dedup                       Load vcbs_embeddings.npy (or embed here)
Clean + normalize                    embed_query()
Save vcbs_reports.json        →      cosine similarity
                                     topK
                                     LLM answer
                                     Gradio UI
                                     
                  ```



In [ ]:
!pip install pdf-inspector
!pip install gradio
!pip install rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 22.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import json
import os
import re
from google.colab import drive, userdata
from sklearn.metrics.pairwise import cosine_similarity
import requests
from pdf_inspector import extract_pages_markdown_bytes
from tqdm import tqdm



In [ ]:
drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive/RAG_Project"
CACHE_FILE = f"{BASE_DIR}/vcbs_embeddings.npy"
OUTPUT_FILE = f"{BASE_DIR}/vcbs_reports.json"
PDF_CACHE_DIR = f"{BASE_DIR}/pdf_cache"
os.makedirs(PDF_CACHE_DIR, exist_ok=True)

with open(f"{BASE_DIR}/vcbs_reports.json", "r", encoding="utf-8") as f:
    vcbs_reports = json.load(f)

doc_ids = [str(r["id"]) for r in vcbs_reports]
doc_texts = [f"{r['name']}. {r['description']}" for r in vcbs_reports]

print(f"Loaded {len(doc_ids)} VCBS reports")

Mounted at /content/drive
Loaded 1000 VCBS reports


In [ ]:
!pip install FlagEmbedding

from FlagEmbedding import BGEM3FlagModel

# Load once — this takes ~30s on first run, cached after
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
EMBED_DIM = 1024
print(f"BGE-M3 loaded, dim={EMBED_DIM}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 loaded, dim=1024


In [ ]:
from FlagEmbedding import FlagReranker
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)
print("Reranker loaded")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker loaded


In [ ]:
# Gemini
import os
from google.colab import userdata
import google.genai as genai

# Securely fetch the key from Colab Secrets
api_key = userdata.get('gemini-key')

# Initialize the client (this replaces genai.configure)
client = genai.Client(api_key=api_key)

In [ ]:
# Fixed Chunking
def chunk_text(text, chunk_size=500, overlap=50):
    if not text or len(text) <= chunk_size:
        return [text] if text else []
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start += chunk_size - overlap
    return chunks

chunk_ids = []
chunk_texts = []
chunk_to_report = {}

for r in vcbs_reports:
    name = r.get("name") or ""
    desc = r.get("description") or ""
    symbol = str(r.get("stockSymbol") or "")
    cat = r.get("category")
    cat_name = cat["name"] if isinstance(cat, dict) else str(cat) if cat else ""

    full_text = f"{name} | {symbol} | {cat_name} | {desc}"

    chunks = chunk_text(full_text, chunk_size=500, overlap=50)
    if not chunks:
        chunks = [f"{name} | {symbol} | {cat_name}"]

    for j, chunk in enumerate(chunks):
        cid = f"{r['id']}_c{j}"
        chunk_ids.append(cid)
        chunk_texts.append(chunk)
        chunk_to_report[cid] = str(r["id"])

print(f"Reports: {len(vcbs_reports)} → Chunks: {len(chunk_ids)}")

Reports: 1000 → Chunks: 3258


In [ ]:
# # Openrouter
# import time
# from openai import OpenAI
# import numpy as np

# # Nemotron client (set up once)
# open_key = userdata.get('openrouter-key')
# openrouter_client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=open_key
# )

In [ ]:
# # Clear Cache

# import os
# if os.path.exists(CACHE_FILE):
#     os.remove(CACHE_FILE)
#     print("Deleted old embeddings — re-run the embedding cell")

LOAD VCBS REPORTS

In [ ]:
df = pd.DataFrame(vcbs_reports)
df["article_id"] = df["id"].astype(str)
df["title"] = df["name"]
print(f"DataFrame: {len(df)} rows")
print(df.columns.tolist())

DataFrame: 1000 rows
['id', 'name', 'stockSymbol', 'category', 'description', 'publishedAt', 'pdf_url', 'locale', 'article_id', 'title']


In [ ]:
for r in vcbs_reports:
    name = r.get("name", "").lower()
    if any(kw in name for kw in ["trái phiếu", "fixed income", "bond", "tpdn"]):
        print(r["name"], "|", r.get("category", ""), "|", r.get("publishedAt", "")[:10])

Fixed income report July - Outlook August 2026 | {'code': 'BCTP', 'name': 'Fixed Income market report'} | 2026-08-11
Báo cáo trái phiếu tháng 7.2026 - Triển vọng 8.2026 | {'code': 'BCTP', 'name': 'Báo cáo trái phiếu'} | 2026-08-07
Corporate Bond - Outlook 2H.2026 | {'code': 'BCTP', 'name': 'Fixed Income market report'} | 2026-07-31
Fixed income report 1H.2026 - Outlook 2H.2026 | {'code': 'BCTP', 'name': 'Fixed Income market report'} | 2026-07-29
Báo cáo Trái phiếu Doanh nghiệp 1H.2026 - Triển Vọng 2H 2026 | {'code': 'BCTP', 'name': 'Báo cáo trái phiếu'} | 2026-07-27
Báo cáo trái phiếu 1H.2026 - Triển vọng 2H.2026 | {'code': 'BCTP', 'name': 'Báo cáo trái phiếu'} | 2026-07-27
Fixed income report May - Outlook June 2026 | {'code': 'BCTP', 'name': 'Fixed Income market report'} | 2026-06-10
Báo cáo trái phiếu tháng 5.2026 - Triển vọng 6.2026 | {'code': 'BCTP', 'name': 'Báo cáo trái phiếu'} | 2026-06-09
Fixed income report April - Outlook May 2026 | {'code': 'BCTP', 'name': 'Fixed Income mar

In [ ]:
# Embed Chunks instead of text
import hashlib

CHUNK_CACHE = f"{BASE_DIR}/vcbs_chunk_embeddings.npy"
CHUNK_HASH_FILE = f"{BASE_DIR}/vcbs_chunk_hash.txt"

# Hash current chunks to detect changes
current_hash = hashlib.md5(f"{len(chunk_texts)}_{chunk_texts[0]}_{chunk_texts[-1]}".encode()).hexdigest()

needs_reembed = True
if os.path.exists(CHUNK_CACHE) and os.path.exists(CHUNK_HASH_FILE):
    with open(CHUNK_HASH_FILE, "r") as f:
        saved_hash = f.read().strip()
    if saved_hash == current_hash:
        chunk_embeddings = np.load(CHUNK_CACHE)
        if chunk_embeddings.shape[0] == len(chunk_texts):
            needs_reembed = False
            print(f"Cache valid — loaded {chunk_embeddings.shape}")

if needs_reembed:
    print(f"Chunks changed ({len(chunk_texts)}) — re-embedding...")
    batch_size = 32
    all_embeddings = []
    for i in range(0, len(chunk_texts), batch_size):
        batch = chunk_texts[i:i+batch_size]
        output = model.encode(batch)["dense_vecs"]
        all_embeddings.append(output)
        if (i // batch_size) % 10 == 0:
            print(f"  Batch {i//batch_size + 1}/{(len(chunk_texts)-1)//batch_size + 1}")
    chunk_embeddings = np.vstack(all_embeddings)
    np.save(CHUNK_CACHE, chunk_embeddings)
    with open(CHUNK_HASH_FILE, "w") as f:
        f.write(current_hash)
    print(f"Saved: {chunk_embeddings.shape}")

print(f"Synced: chunks={len(chunk_texts)} embeddings={chunk_embeddings.shape[0]}")

Chunks changed (3258) — re-embedding...
  Batch 1/102
  Batch 11/102
  Batch 21/102
  Batch 31/102
  Batch 41/102
  Batch 51/102
  Batch 61/102
  Batch 71/102
  Batch 81/102
  Batch 91/102
  Batch 101/102
Saved: (3258, 1024)
Synced: chunks=3258 embeddings=3258


In [ ]:
# Embed query

def embed_query(text):
    output = model.encode([text])["dense_vecs"]
    return np.array(output)

In [ ]:
# BM25 Index Score for Chunk Texts
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r'\w+', text.lower())

bm25_corpus = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(bm25_corpus)
print(f"BM25 index built: {len(bm25_corpus)} chunks")

BM25 index built: 3258 chunks


In [ ]:
def hybrid_search(query, top_n=20, alpha=0.4):
    q_embedding = embed_query(query)

    dense_scores = cosine_similarity(q_embedding, chunk_embeddings).ravel()
    bm25_scores_raw = bm25.get_scores(tokenize(query))

    def normalize(scores):
        mn, mx = scores.min(), scores.max()
        if mx == mn:
            return np.zeros_like(scores)
        return (scores - mn) / (mx - mn)

    dense_norm = normalize(dense_scores)
    bm25_norm = normalize(bm25_scores_raw)

    fused = alpha * bm25_norm + (1 - alpha) * dense_norm

    # Aggregate chunks → reports
    report_scores = {}
    report_dense = {}
    report_bm25 = {}

    for i, cid in enumerate(chunk_ids):
        rid = chunk_to_report[cid]
        if rid not in report_scores or fused[i] > report_scores[rid]:
            report_scores[rid] = float(fused[i])
            report_dense[rid] = float(dense_norm[i])
            report_bm25[rid] = float(bm25_norm[i])

    sorted_reports = sorted(report_scores.items(), key=lambda x: x[1], reverse=True)

    # Build results with method breakdown
    results = []
    for rid, score in sorted_reports[:top_n]:
        d = report_dense.get(rid, 0)
        b = report_bm25.get(rid, 0)
        dominant = "BM25" if b > d else "Dense" if d > b else "Equal"
        results.append((rid, score, d, b, dominant))

    return results

In [ ]:
def rerank_results(query, search_results, top_n=5):
  # OG
    # lookup = {str(r["id"]): r for r in vcbs_reports}

    # pairs = []
    # for rid, score, d, b, dom in search_results:
    #     r = lookup.get(rid, {})
    #     text = f"{r.get('name', '')} | {r.get('description', '')}"
    #     pairs.append([query, text])

    # ce_scores = reranker.compute_score(pairs)

    # scored = list(zip(search_results, ce_scores))
    # scored.sort(key=lambda x: x[1], reverse=True)

    # return [item[0] for item in scored[:top_n]]

    lookup = {str(r["id"]): r for r in vcbs_reports}
    pairs = []
    for rid, score, d, b, dom in search_results:
        r = lookup.get(rid, {})
        symbol = str(r.get("stockSymbol") or "")
        cat = r.get("category")
        cat_name = cat["name"] if isinstance(cat, dict) else str(cat) if cat else ""
        text = f"{r.get('name', '')} | {symbol} | {cat_name} | {r.get('description', '')[:500]}"
        pairs.append([query, text])
    ce_scores = reranker.compute_score(pairs)
    scored = list(zip(search_results, ce_scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [item[0] for item in scored[:top_n]]

In [ ]:
# Download PDF -> extract text thru pdf-inspector -> cach document
PDF_CACHE_DIR = f"{BASE_DIR}/pdf_cache"
os.makedirs(PDF_CACHE_DIR, exist_ok=True)

def download_and_extract(pdf_url, max_pages=5):
    # Check cache first
    cache_key = pdf_url.split("/")[-1].replace(".pdf", ".txt")
    cache_path = f"{PDF_CACHE_DIR}/{cache_key}"

    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            return f.read()

    try:
        res = requests.get(pdf_url, timeout=30)
        res.raise_for_status()

        result = extract_pages_markdown_bytes(res.content)
        text = "\n\n".join(
            page.markdown for page in result.pages[:max_pages]
            if page.markdown
        )
        text = text.strip()

        # Save to cache
        if text:
            with open(cache_path, "w", encoding="utf-8") as f:
                f.write(text)

        return text

    except Exception as e:
        print(f"PDF failed: {pdf_url[:60]}... → {e}")
        return ""

In [ ]:
import re
from datetime import datetime, timedelta

def parse_time_filter(query):
    q = query.lower()
    today = datetime.now()

    if "today" in q or "hôm nay" in q:
        return today.strftime("%Y-%m-%d"), today.strftime("%Y-%m-%d")
    if "this week" in q or "tuần này" in q:
        start = today - timedelta(days=today.weekday())
        return start.strftime("%Y-%m-%d"), today.strftime("%Y-%m-%d")
    if "this month" in q or "tháng này" in q:
        return today.strftime("%Y-%m-01"), today.strftime("%Y-%m-%d")
    if "latest" in q or "mới nhất" in q or "gần nhất" in q:
        start = today - timedelta(days=14)
        return start.strftime("%Y-%m-%d"), today.strftime("%Y-%m-%d")

    # Match "March 2026", "tháng 3 2026", "08/2026"
    m = re.search(r'(january|february|march|april|may|june|july|august|september|october|november|december)\s*(\d{4})', q)
    if m:
        months = {"january":"01","february":"02","march":"03","april":"04",
                  "may":"05","june":"06","july":"07","august":"08",
                  "september":"09","october":"10","november":"11","december":"12"}
        mon = months[m.group(1)]
        year = m.group(2)
        return f"{year}-{mon}-01", f"{year}-{mon}-31"

    m = re.search(r'(\d{1,2})/(\d{4})', q)
    if m:
        return f"{m.group(2)}-{m.group(1).zfill(2)}-01", f"{m.group(2)}-{m.group(1).zfill(2)}-31"

    return None, None

In [ ]:
def is_clean_text(text, min_ratio=0.7):
    """Reject text where less than 70% of chars are normal letters/numbers/punctuation."""
    if not text:
        return False
    clean_chars = sum(1 for c in text if c.isalnum() or c.isspace() or c in '.,;:!?()-/%')
    return (clean_chars / len(text)) > min_ratio

In [ ]:
# Build context from search result
def build_context(search_results, query):
    lookup = {str(r["id"]): r for r in vcbs_reports}
    context = ""
    result_text = ""
    pdf_count = 0

    for rank, (rid, score, dense_s, bm25_s, dominant) in enumerate(search_results, 1):
        r = lookup.get(rid, {})
        title = r.get("name", "N/A")
        desc = r.get("description") or ""
        pub_date = r.get("publishedAt") or ""
        pdf_url = r.get("pdf_url", "")

        result_text += f"Rank {rank} (fused: {score:.3f} | dense: {dense_s:.3f} | bm25: {bm25_s:.3f} | won: {dominant})\n"
        result_text += f"  {pub_date[:10] if pub_date else 'N/A'} - {title}\n"
        result_text += f"  {desc[:150] if desc else '(no description)'}\n\n"

        if pdf_url and len(desc) < 200 and pdf_count < 2:
            pdf_text = download_and_extract(pdf_url)
            if pdf_text and is_clean_text(pdf_text):  # ← add quality check
                pdf_count += 1
                context += f"[{rank}] {title} ({pub_date[:10] if pub_date else ''})\n{pdf_text[:3000]}\n\n"
                continue
        context += f"[{rank}] {title} ({pub_date[:10] if pub_date else ''})\n{desc}\n\n"

    return result_text, context[:12000]

In [ ]:
from google.genai import types

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_reports",
            description="Search VCBS financial reports database. Use this when the user asks about Vietnamese stocks, market analysis, bonds, specific tickers, or any financial data. Do NOT use for general knowledge questions like 'what is a stock'.",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": "The search query to find relevant reports. Can be in English or Vietnamese."
                    }
                },
                "required": ["query"]
            }
        )
    ]
)

In [ ]:
def execute_search(query):
    date_from, date_to = parse_time_filter(query)
    results = hybrid_search(query, top_n=10, alpha=0.4)

    if date_from:
        lookup = {str(r["id"]): r for r in vcbs_reports}
        filtered = [(rid, s, d, b, dom) for rid, s, d, b, dom in results
                    if date_from <= (lookup.get(rid, {}).get("publishedAt") or "")[:10] <= date_to]
        if filtered:
            results = filtered

    # Just return metadata — NO PDF downloads during search
    lookup = {str(r["id"]): r for r in vcbs_reports}
    context = ""
    for rank, (rid, score, d, b, dom) in enumerate(results[:5], 1):
        r = lookup.get(rid, {})
        title = r.get("name", "N/A")
        desc = r.get("description") or ""
        pub_date = r.get("publishedAt") or ""
        if desc and is_clean_text(desc):
            context += f"[{rank}] {title} ({pub_date[:10] if pub_date else ''})\n{desc[:500]}\n\n"
        else:
            context += f"[{rank}] {title} ({pub_date[:10] if pub_date else ''})\n(no description)\n\n"

    return context

In [ ]:
from datetime import date

def agentic_rag(user_query):
    if not user_query or not user_query.strip():
        return "Please enter a question.", ""

    try:
        system_prompt = f"""You are a financial research assistant with access to a VCBS reports database. They are only documentations
        reports, so take them in, and answer accordingly. Follow the rules as tightly as possible.
Today's date is {date.today()}.

Rules:
- For questions about Vietnamese markets, stocks, bonds: call search_reports for EACH topic separately with a clear, comprehensive query
- For general knowledge questions: answer directly without searching
- After receiving search results: synthesize a comprehensive answer citing document titles
- IMPORTANT: Always respond in the SAME language the user typed their question in. If the question is in English, respond entirely in English even if the search results are in Vietnamese."""

        messages = [
            types.Content(role="user", parts=[types.Part(text=f"{system_prompt}\n\nUser question: {user_query}")])
        ]

        all_result_text = ""
        max_rounds = 2

        for round_num in range(max_rounds):
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=messages,
                config=types.GenerateContentConfig(tools=[search_tool])
            )

            has_function_call = False
            function_parts = []

            for part in response.candidates[0].content.parts:
                if part.function_call:
                    has_function_call = True
                    fn_name = part.function_call.name
                    fn_args = part.function_call.args
                    rephrased = fn_args.get("query", user_query)
                    print(f"  Round {round_num+1}: Gemini calls {fn_name}")
                    print(f"    Original:  {user_query[:80]}")
                    print(f"    Rephrased: {rephrased[:80]}")

                    # --- DUAL QUERY: both original + rephrased ---
                    results_original = hybrid_search(user_query, top_n=15)
                    results_rephrased = hybrid_search(rephrased, top_n=15)

                    # Merge: keep best score per report ID
                    merged = {}
                    for rid, score, d, b, dom in results_original + results_rephrased:
                        if rid not in merged or score > merged[rid][1]:
                            merged[rid] = (rid, score, d, b, dom)

                    combined = sorted(merged.values(), key=lambda x: x[1], reverse=True)[:20]
                    combined = rerank_results(user_query, combined, top_n=10)

                    # Time filter (applies to merged results)
                    date_from, date_to = parse_time_filter(user_query)
                    if date_from:
                        lookup = {str(r["id"]): r for r in vcbs_reports}
                        filtered = [(rid, s, d, b, dom) for rid, s, d, b, dom in combined
                                    if date_from <= (lookup.get(rid, {}).get("publishedAt") or "")[:10] <= date_to]
                        if filtered:
                            combined = filtered

                    result_text, context = build_context(combined, user_query)
                    all_result_text = result_text

                    function_parts.append(
                        types.Part(function_response=types.FunctionResponse(
                            name=fn_name,
                            response={"results": context}
                        ))
                    )

            if has_function_call:
                messages.append(response.candidates[0].content)
                messages.append(types.Content(role="user", parts=function_parts))
            else:
                final_answer = response.candidates[0].content.parts[0].text
                if not all_result_text:
                    return "", final_answer
                return all_result_text, final_answer

        # If max rounds hit, force a final text response
        final_response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=messages
        )
        final_answer = final_response.candidates[0].content.parts[0].text
        return all_result_text, final_answer

    except Exception as e:
        import traceback
        return f"Error: {traceback.format_exc()}", str(e)

In [ ]:
import gradio as gr

with gr.Blocks(title="VCBS Agentic RAG") as demo:
    gr.Markdown("# VCBS Agentic RAG Search")
    gr.Markdown("*Gemini decides whether to search, rephrases your query, and compiles results*")
    query_input = gr.Textbox(label="Your question", placeholder="e.g. HPG steel outlook, báo cáo trái phiếu mới nhất")
    search_btn = gr.Button("Ask")

    with gr.Accordion("Thinking — Retrieved Documents", open=False):
        results_output = gr.Textbox(label="Documents", lines=12)

    answer_output = gr.Markdown(label="AI Answer")

    search_btn.click(
        fn=agentic_rag,
        inputs=query_input,
        outputs=[results_output, answer_output]
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c69fb6f5ee2ba56db1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c69fb6f5ee2ba56db1.gradio.live


EVALUATE

In [ ]:
def find_ids(*keywords):
    name_matches = []
    other_matches = []
    for r in vcbs_reports:
        name = (r.get("name") or "").lower()
        symbol = str(r.get("stockSymbol") or "").lower()
        desc = (r.get("description") or "").lower()

        if all(k.lower() in name or k.lower() in symbol for k in keywords):
            name_matches.append(str(r["id"]))
        elif all(k.lower() in f"{name} {desc} {symbol}" for k in keywords):
            other_matches.append(str(r["id"]))

    return (name_matches + other_matches)[:3]

In [ ]:
eval_dataset = [
    {"query": "HPG steel outlook 2025", "difficulty": "easy",
     "relevant_doc_ids": find_ids("HPG"), "expected_answer": "Hoa Phat steel analysis"},
    {"query": "FPT technology earnings report", "difficulty": "easy",
     "relevant_doc_ids": find_ids("FPT"), "expected_answer": "FPT earnings report"},
    {"query": "VCB banking results", "difficulty": "easy",
     "relevant_doc_ids": find_ids("VCB"), "expected_answer": "Vietcombank results"},
    {"query": "VNM dairy industry analysis", "difficulty": "easy",
     "relevant_doc_ids": find_ids("VNM"), "expected_answer": "Vinamilk analysis"},
    {"query": "MWG retail performance", "difficulty": "easy",
     "relevant_doc_ids": find_ids("MWG"), "expected_answer": "Mobile World performance"},

    {"query": "Compare FPT and VCB earnings", "difficulty": "medium",
     "relevant_doc_ids": find_ids("FPT") + find_ids("VCB"), "expected_answer": "FPT vs VCB comparison"},
    {"query": "Vietnam banking sector risks 2025", "difficulty": "medium",
     "relevant_doc_ids": find_ids("bank"), "expected_answer": "Banking sector risk analysis"},
    {"query": "Steel and construction stocks outlook", "difficulty": "medium",
     "relevant_doc_ids": find_ids("steel") + find_ids("construction"), "expected_answer": "Steel/construction outlook"},
    {"query": "Fixed income bond market weekly", "difficulty": "medium",
     "relevant_doc_ids": find_ids("fixed", "income"), "expected_answer": "Bond market weekly report"},
    {"query": "Vietnam market investment strategy", "difficulty": "medium",
     "relevant_doc_ids": find_ids("strategy"), "expected_answer": "Investment strategy"},

    {"query": "Which sector will outperform if interest rates rise", "difficulty": "hard",
     "relevant_doc_ids": find_ids("interest") or find_ids("bank"), "expected_answer": "Rate-sensitive sectors"},
    {"query": "Impact of global trade on Vietnamese exports", "difficulty": "hard",
     "relevant_doc_ids": find_ids("export") or find_ids("trade"), "expected_answer": "Trade impact analysis"},
    {"query": "Best long term investment strategy Vietnam 2025", "difficulty": "hard",
     "relevant_doc_ids": find_ids("strategy") or find_ids("outlook"), "expected_answer": "Long-term strategy"},

    {"query": "What should I invest in", "difficulty": "ambiguous",
     "relevant_doc_ids": [], "expected_answer": "General advice"},
    {"query": "Is the stock market risky", "difficulty": "ambiguous",
     "relevant_doc_ids": [], "expected_answer": "General risk statement"},
]

for q in eval_dataset:
    print(f"[{q['difficulty']:8s}] {len(q['relevant_doc_ids'])} docs | {q['query'][:50]}")

[easy    ] 3 docs | HPG steel outlook 2025
[easy    ] 3 docs | FPT technology earnings report
[easy    ] 3 docs | VCB banking results
[easy    ] 3 docs | VNM dairy industry analysis
[easy    ] 3 docs | MWG retail performance
[medium  ] 6 docs | Compare FPT and VCB earnings
[medium  ] 3 docs | Vietnam banking sector risks 2025
[medium  ] 6 docs | Steel and construction stocks outlook
[medium  ] 3 docs | Fixed income bond market weekly
[medium  ] 3 docs | Vietnam market investment strategy
[hard    ] 3 docs | Which sector will outperform if interest rates ris
[hard    ] 3 docs | Impact of global trade on Vietnamese exports
[hard    ] 3 docs | Best long term investment strategy Vietnam 2025
[ambiguous] 0 docs | What should I invest in
[ambiguous] 0 docs | Is the stock market risky


In [ ]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    if not relevant_ids:
        return 1.0
    top_k = retrieved_ids[:k]
    found = len(set(top_k) & set(relevant_ids))
    return found / len(relevant_ids)

def precision_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    if not top_k:
        return 0.0
    found = len(set(top_k) & set(relevant_ids))
    return found / len(top_k)

def mrr(retrieved_ids, relevant_ids):
    if not relevant_ids:
        return 0.0
    for i, rid in enumerate(retrieved_ids):
        if rid in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(retrieved_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0
    top_k = retrieved_ids[:k]
    dcg = sum(1.0 / np.log2(i + 2) for i, rid in enumerate(top_k) if rid in relevant_ids)
    ideal = sorted([1] * min(len(relevant_ids), k) + [0] * max(0, k - len(relevant_ids)), reverse=True)
    idcg = sum(val / np.log2(i + 2) for i, val in enumerate(ideal) if val > 0)
    return dcg / idcg if idcg > 0 else 0.0

def hit_rate(retrieved_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0
    top_k = retrieved_ids[:k]
    return 1.0 if set(top_k) & set(relevant_ids) else 0.0

In [ ]:
def run_retrieval(query, mode="full", top_n=10, alpha=0.4):
    if mode == "bm25_only":
        bm25_scores_raw = bm25.get_scores(tokenize(query))
        ranked = np.argsort(bm25_scores_raw)[::-1][:top_n * 3]
        report_ids = []
        seen = set()
        for idx in ranked:
            rid = chunk_to_report[chunk_ids[idx]]
            if rid not in seen:
                seen.add(rid)
                report_ids.append(rid)
        return report_ids[:top_n]

    elif mode == "dense_only":
        q_emb = embed_query(query)
        dense_scores = cosine_similarity(q_emb, chunk_embeddings).ravel()
        ranked = np.argsort(dense_scores)[::-1][:top_n * 3]
        report_ids = []
        seen = set()
        for idx in ranked:
            rid = chunk_to_report[chunk_ids[idx]]
            if rid not in seen:
                seen.add(rid)
                report_ids.append(rid)
        return report_ids[:top_n]

    elif mode == "hybrid":
        results = hybrid_search(query, top_n=top_n, alpha=alpha)
        return [r[0] for r in results]

    elif mode == "hybrid_reranker":
        results = hybrid_search(query, top_n=20, alpha=alpha)
        results = rerank_results(query, results, top_n=top_n)
        return [r[0] for r in results]

In [ ]:
import random
random.seed(42)

templates = [
    "{symbol} {cat} report",
    "{symbol} outlook",
    "{symbol} latest update",
    "{symbol} {cat} analysis",
    "What is the outlook for {symbol}",
    "{symbol} earnings and performance",
]

candidates = [r for r in vcbs_reports if r.get("stockSymbol")]
sample_reports = random.sample(candidates, min(50, len(candidates)))

auto_eval_dataset = []
for r in sample_reports:
    symbol = r["stockSymbol"]
    cat = r.get("category")
    cat_name = cat["name"] if isinstance(cat, dict) else str(cat) if cat else "report"

    template = random.choice(templates)
    query = template.format(symbol=symbol, cat=cat_name)

    auto_eval_dataset.append({
        "query": query,
        "difficulty": "auto",
        "relevant_doc_ids": [str(r["id"])],
        "expected_answer": r.get("name", "")
    })

eval_dataset = eval_dataset + auto_eval_dataset
print(f"Total eval queries: {len(eval_dataset)}")

Total eval queries: 65


In [ ]:
import time
import pandas as pd

K = 10
modes = ["bm25_only", "dense_only", "hybrid", "hybrid_reranker"]
results_table = []
retrieval_queries = [q for q in eval_dataset if q["relevant_doc_ids"]]

for mode in modes:
    scores = {"recall": [], "precision": [], "mrr": [], "ndcg": [], "hit": []}
    start = time.time()

    for i, q in enumerate(retrieval_queries):
        print(f"  {mode} [{i+1}/{len(retrieval_queries)}]", end="\r")
        retrieved = run_retrieval(q["query"], mode=mode, top_n=K)
        relevant = [str(rid) for rid in q["relevant_doc_ids"]]

        scores["recall"].append(recall_at_k(retrieved, relevant, K))
        scores["precision"].append(precision_at_k(retrieved, relevant, K))
        scores["mrr"].append(mrr(retrieved, relevant))
        scores["ndcg"].append(ndcg_at_k(retrieved, relevant, K))
        scores["hit"].append(hit_rate(retrieved, relevant, K))

    elapsed = time.time() - start
    row = {
        "Mode": mode,
        "Recall@10": f"{np.mean(scores['recall']):.3f}",
        "Precision@10": f"{np.mean(scores['precision']):.3f}",
        "MRR": f"{np.mean(scores['mrr']):.3f}",
        "NDCG@10": f"{np.mean(scores['ndcg']):.3f}",
        "HitRate@10": f"{np.mean(scores['hit']):.3f}",
        "Time(s)": f"{elapsed:.1f}",
    }
    results_table.append(row)
    print(f"Done: {mode} ({elapsed:.1f}s)")

# outside the loop now
ablation_df = pd.DataFrame(results_table)
ablation_df

Done: bm25_only (0.4s)
Done: dense_only (6.5s)
Done: hybrid (6.2s)
Done: hybrid_reranker (27.0s)


,Mode,Recall@10,Precision@10,MRR,NDCG@10,HitRate@10,Time(s)
0,bm25_only,0.640,0.071,0.409,0.456,0.667,0.4
1,dense_only,0.839,0.100,0.503,0.584,0.857,6.5
2,hybrid,0.865,0.102,0.644,0.689,0.889,6.2
3,hybrid_reranker,0.870,0.103,0.627,0.689,0.905,27.0


In [ ]:
def analyze_errors(eval_dataset, mode="hybrid_reranker", k=10, deep_k=50):
    errors = []
    for q in eval_dataset:
        relevant = [str(rid) for rid in q["relevant_doc_ids"]]
        if not relevant:
            continue  # skip ambiguous/no-RAG queries — nothing to fail

        retrieved = run_retrieval(q["query"], mode=mode, top_n=k)
        hit = bool(set(retrieved) & set(relevant))
        if hit:
            continue  # success, no error to log

        # Dig deeper: does the doc exist further down?
        deep_results = hybrid_search(q["query"], top_n=deep_k)
        deep_ids = [r[0] for r in deep_results]
        found_deep = set(deep_ids) & set(relevant)

        if found_deep:
            ranks = [deep_ids.index(rid) + 1 for rid in found_deep]
            error_type = "Bad Ranking"
            detail = f"Found at rank(s) {ranks} within top {deep_k}, but fell outside top {k}"
        else:
            error_type = "Missing Document"
            detail = f"Not found even within top {deep_k} candidates"

        errors.append({
            "query": q["query"],
            "difficulty": q["difficulty"],
            "relevant_ids": relevant,
            "retrieved_top10": retrieved,
            "error_type": error_type,
            "detail": detail,
        })

    return errors

error_log = analyze_errors(eval_dataset, mode="hybrid_reranker")
total_with_gt = len([q for q in eval_dataset if q["relevant_doc_ids"]])
print(f"Failures: {len(error_log)} / {total_with_gt} queries with ground truth")

error_df = pd.DataFrame(error_log)
error_df

print(error_df["error_type"].value_counts())
print("\nFailures by difficulty:")
print(error_df["difficulty"].value_counts())


Failures: 6 / 63 queries with ground truth
error_type
Missing Document    3
Bad Ranking         3
Name: count, dtype: int64

Failures by difficulty:
difficulty
medium    3
hard      2
easy      1
Name: count, dtype: int64


In [ ]:
pd.set_option("display.max_colwidth", None)
for _, row in error_df.iterrows():
    print(f"[{row['difficulty']}] {row['error_type']}")
    print(f"  Query: {row['query']}")
    print(f"  Expected: {row['relevant_ids']}")
    print(f"  Got top10: {row['retrieved_top10']}")
    print(f"  {row['detail']}")
    print()

[easy] Missing Document
  Query: VCB banking results
  Expected: ['13962', '13961', '13960']
  Got top10: ['12918', '12919', '12923', '13323', '12948', '13329', '12809', '12826', '12794', '12773']
  Not found even within top 50 candidates

[medium] Missing Document
  Query: Vietnam banking sector risks 2025
  Expected: ['13688', '13339', '13153']
  Got top10: ['12938', '13130', '13133', '12831', '12774', '12948', '13078', '13394', '12969', '13399']
  Not found even within top 50 candidates

[medium] Bad Ranking
  Query: Steel and construction stocks outlook
  Expected: ['13951', '13679', '13644', '12791', '13947', '13933']
  Got top10: ['12981', '12985', '12794', '13017', '13237', '12782', '13490', '12923', '12957', '13475']
  Found at rank(s) [45, 26, 16] within top 50, but fell outside top 10

[medium] Bad Ranking
  Query: Fixed income bond market weekly
  Expected: ['13953', '13934', '13916']
  Got top10: ['13830', '13337', '12839', '12856', '13058', '13707', '12943', '13018', '1340

In [ ]:
from IPython.display import Markdown, display
from collections import Counter

difficulty_counts = Counter(q["difficulty"] for q in eval_dataset)
error_type_counts = Counter(e["error_type"] for e in error_log)

overview = f"""
# VCBS Agentic RAG: Project Report

## 1. Pipeline Architecture
- **Corpus**: {len(vcbs_reports)} VCBS financial reports (English + Vietnamese)
- **Chunking**: {len(chunk_ids)} chunks (fixed-size, 500 chars / 50 overlap)
- **Embedding**: BGE-M3 (1024-dim, multilingual, fp16 on T4)
- **Retrieval**: Hybrid search — BM25 + Dense cosine similarity, min-max normalized, α=0.4 weighted fusion
- **Reranking**: BGE cross-encoder (bge-reranker-v2-m3), top 20 → top 10
- **Orchestration**: Gemini agentic function calling — decides whether to search, rephrases query, dual-query merge (original + rephrased), time-based metadata filtering
- **Interface**: Gradio UI with collapsible "Thinking" panel showing retrieval breakdown

## 2. Evaluation Dataset
**{len(eval_dataset)} queries** across difficulty tiers:
"""
for level, count in difficulty_counts.items():
    overview += f"- **{level}**: {count} queries\n"

overview += "\n## 3. Retrieval Metrics Implemented\nRecall@K, Precision@K, MRR, NDCG@K, Hit Rate@K\n\n## 4. Baseline + Ablation Study"

display(Markdown(overview))
display(ablation_df)

analysis = """
**Key finding**: Hybrid fusion delivers the largest single quality jump over single-method baselines
(MRR +28% vs dense-only). Reranking adds modest recall/hit-rate gains at ~4x the latency cost,
with a slight MRR regression — a genuine quality/latency tradeoff, not a defect.

## 5. Error Analysis
"""
total_with_gt = len([q for q in eval_dataset if q["relevant_doc_ids"]])
analysis += f"- **{len(error_log)} failures** out of {total_with_gt} queries with ground truth ({(1 - len(error_log)/total_with_gt)*100:.1f}% hit rate)\n"
for etype, count in error_type_counts.items():
    analysis += f"- **{etype}**: {count}\n"

analysis += """
**Root cause breakdown**: 3/6 failures are genuine ranking weaknesses on compound or abstract queries
(correct docs retrieved but ranked outside top 10). 3/6 trace back to eval-label noise from substring
collisions in automated ground-truth generation (e.g. ticker "VCB" matching the unrelated report-title
prefix "VCBS"). The pipeline's true accuracy is likely understated by the raw failure count.

## 6. Known Limitations / Future Work
- LLM-as-judge evaluation (groundedness, answer correctness) not yet implemented
- No formal multi-run experiment-tracking log
- Metadata filtering covers time only; category/locale filtering not implemented
- Eval ground truth for hand-written queries built via keyword search; not manually verified
"""
display(Markdown(analysis))
display(error_df)


# VCBS Agentic RAG: Project Report

## 1. Pipeline Architecture
- **Corpus**: 1000 VCBS financial reports (English + Vietnamese)
- **Chunking**: 3258 chunks (fixed-size, 500 chars / 50 overlap)
- **Embedding**: BGE-M3 (1024-dim, multilingual, fp16 on T4)
- **Retrieval**: Hybrid search — BM25 + Dense cosine similarity, min-max normalized, α=0.4 weighted fusion
- **Reranking**: BGE cross-encoder (bge-reranker-v2-m3), top 20 → top 10
- **Orchestration**: Gemini agentic function calling — decides whether to search, rephrases query, dual-query merge (original + rephrased), time-based metadata filtering
- **Interface**: Gradio UI with collapsible "Thinking" panel showing retrieval breakdown

## 2. Evaluation Dataset
**65 queries** across difficulty tiers:
- **easy**: 5 queries
- **medium**: 5 queries
- **hard**: 3 queries
- **ambiguous**: 2 queries
- **auto**: 50 queries

## 3. Retrieval Metrics Implemented
Recall@K, Precision@K, MRR, NDCG@K, Hit Rate@K

## 4. Baseline + Ablation Study

,Mode,Recall@10,Precision@10,MRR,NDCG@10,HitRate@10,Time(s)
0,bm25_only,0.640,0.071,0.409,0.456,0.667,0.4
1,dense_only,0.839,0.100,0.503,0.584,0.857,6.5
2,hybrid,0.865,0.102,0.644,0.689,0.889,6.2
3,hybrid_reranker,0.870,0.103,0.627,0.689,0.905,27.0



**Key finding**: Hybrid fusion delivers the largest single quality jump over single-method baselines
(MRR +28% vs dense-only). Reranking adds modest recall/hit-rate gains at ~4x the latency cost,
with a slight MRR regression — a genuine quality/latency tradeoff, not a defect.

## 5. Error Analysis
- **6 failures** out of 63 queries with ground truth (90.5% hit rate)
- **Missing Document**: 3
- **Bad Ranking**: 3

**Root cause breakdown**: 3/6 failures are genuine ranking weaknesses on compound or abstract queries
(correct docs retrieved but ranked outside top 10). 3/6 trace back to eval-label noise from substring
collisions in automated ground-truth generation (e.g. ticker "VCB" matching the unrelated report-title
prefix "VCBS"). The pipeline's true accuracy is likely understated by the raw failure count.

## 6. Known Limitations / Future Work
- LLM-as-judge evaluation (groundedness, answer correctness) not yet implemented
- No formal multi-run experiment-tracking log
- Metadata filtering covers time only; category/locale filtering not implemented
- Eval ground truth for hand-written queries built via keyword search; not manually verified


,query,difficulty,relevant_ids,retrieved_top10,error_type,detail
0,VCB banking results,easy,"[13962, 13961, 13960]","[12918, 12919, 12923, 13323, 12948, 13329, 12809, 12826, 12794, 12773]",Missing Document,Not found even within top 50 candidates
1,Vietnam banking sector risks 2025,medium,"[13688, 13339, 13153]","[12938, 13130, 13133, 12831, 12774, 12948, 13078, 13394, 12969, 13399]",Missing Document,Not found even within top 50 candidates
2,Steel and construction stocks outlook,medium,"[13951, 13679, 13644, 12791, 13947, 13933]","[12981, 12985, 12794, 13017, 13237, 12782, 13490, 12923, 12957, 13475]",Bad Ranking,"Found at rank(s) [45, 26, 16] within top 50, but fell outside top 10"
3,Fixed income bond market weekly,medium,"[13953, 13934, 13916]","[13830, 13337, 12839, 12856, 13058, 13707, 12943, 13018, 13409, 13862]",Bad Ranking,"Found at rank(s) [24, 21] within top 50, but fell outside top 10"
4,Which sector will outperform if interest rates rise,hard,"[13961, 13951, 13947]","[13607, 13685, 13654, 13027, 12920, 13529, 12937, 13041, 13756, 13441]",Missing Document,Not found even within top 50 candidates
5,Best long term investment strategy Vietnam 2025,hard,"[13947, 13930, 13723]","[12831, 13394, 13399, 13013, 13055, 13300, 13263, 13283, 13073, 12927]",Bad Ranking,"Found at rank(s) [42] within top 50, but fell outside top 10"
